# SafeDrive: VS Code + Colab connection test

Run this notebook from top to bottom using a Colab kernel in VS Code. It verifies five things without starting RL training:

1. VS Code can execute on a Colab-managed runtime.
2. PyTorch can actually use the assigned GPU.
3. Google Drive can be mounted and written to.
4. The same SafeDrive revision can be cloned from GitHub and installed.
5. MetaDrive can run headlessly on the remote machine.

Authentication prompts must be completed by you. Do not paste Google passwords or GitHub tokens into this notebook.

In [10]:
from pathlib import Path

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "SafeDrive"

print(f"Remote workspace: {REPO_DIR}")
print(f"Persistent storage: {DRIVE_PROJECT}")

Remote workspace: /content/safedrive
Persistent storage: /content/drive/MyDrive/SafeDrive


In [4]:
import platform
import subprocess
import sys
import time

import torch

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
subprocess.run(["nvidia-smi"], check=False)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached. In VS Code choose Select Kernel > Colab > "
        "New Colab Server and select a GPU machine."
    )

gpu_name = torch.cuda.get_device_name(0)
start = time.perf_counter()
left = torch.rand((1024, 1024), device="cuda")
right = torch.rand((1024, 1024), device="cuda")
product = left @ right
torch.cuda.synchronize()
elapsed = time.perf_counter() - start
assert product.is_cuda
print(f"GPU calculation passed on {gpu_name} in {elapsed:.3f} seconds.")

Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
GPU calculation passed on Tesla T4 in 0.387 seconds.


In [6]:
from datetime import datetime, timezone

from google.colab import drive

# Google will display a separate authorization prompt for this mount.
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

drive_probe = DRIVE_PROJECT / "colab_connection_test.txt"
probe_text = f"SafeDrive Colab connection passed at {datetime.now(timezone.utc).isoformat()}\n"
drive_probe.write_text(probe_text, encoding="utf-8")
assert drive_probe.read_text(encoding="utf-8") == probe_text
print(f"Google Drive read/write passed: {drive_probe}")

Mounted at /content/drive
Google Drive read/write passed: /content/drive/MyDrive/SafeDrive/colab_connection_test.txt


## GitHub access

SafeDrive is public, so this notebook clones it without a GitHub token. Push local changes before running this cell when Colab needs the newest revision.

In [9]:
%ls

drive/  sample_data/


In [13]:
import os

if (REPO_DIR / ".git").exists():
    git_command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
elif REPO_DIR.exists() and not any(REPO_DIR.iterdir()):
    REPO_DIR.rmdir()
    git_command = ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)]
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    git_command = ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)]

git_result = subprocess.run(
    git_command,
    check=False,
    capture_output=True,
    text=True,
)
if git_result.returncode:
    git_error = git_result.stderr.strip() or "Git returned no error text."
    raise RuntimeError(f"Git could not update SafeDrive.\n\nGit said:\n{git_error}")
if git_result.stdout.strip():
    print(git_result.stdout.strip())
if git_result.stderr.strip():
    print(git_result.stderr.strip())

os.chdir(REPO_DIR)
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
print(f"SafeDrive revision: {git_commit}")
print(f"Working directory: {Path.cwd()}")

Updating e3f6131..4ba4649
Fast-forward
 README.md                        |   7 +-
 notebooks/colab_smoke_test.ipynb | 158 +++++++++++++++++++++++++++------------
 pyproject.toml                   |   2 +-
 3 files changed, 116 insertions(+), 51 deletions(-)
From https://github.com/djdhillxn/safedrive
   e3f6131..4ba4649  main       -> origin/main
SafeDrive revision: 4ba4649f12937d61302f474b28d7c0b1ab6cf560
Working directory: /content/safedrive


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
import importlib.metadata

# Colab includes unmaintained OpenAI Gym. SafeDrive uses Gymnasium exclusively.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "gym"],
    check=False,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--progress-bar",
        "off",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

try:
    legacy_gym_version = importlib.metadata.version("gym")
except importlib.metadata.PackageNotFoundError:
    print("Legacy Gym removed; SafeDrive will use Gymnasium.")
else:
    raise RuntimeError(f"Legacy Gym is still installed: {legacy_gym_version}")

package_names = [
    "metadrive-simulator",
    "stable-baselines3",
    "sb3-contrib",
    "gymnasium",
]
package_versions = {
    name: importlib.metadata.version(name) for name in package_names
}
for name, version in package_versions.items():
    print(f"{name}: {version}")

import saferl_drive
from stable_baselines3 import PPO, SAC

print(f"SafeDrive import passed: {saferl_drive.__version__}")

Legacy Gym removed; SafeDrive will use Gymnasium.
metadrive-simulator: 0.4.3
stable-baselines3: 2.9.0
sb3-contrib: 2.9.0
gymnasium: 1.3.0
SafeDrive import passed: 0.1.0


In [15]:
from metadrive.envs import MetaDriveEnv

environment = MetaDriveEnv(
    {
        "use_render": False,
        "log_level": 50,
        "num_scenarios": 1,
        "start_seed": 0,
        "traffic_density": 0.0,
        "random_traffic": False,
        "horizon": 50,
    }
)

rollout_reward = 0.0
rollout_steps = 0
try:
    observation, info = environment.reset(seed=0)
    for _ in range(25):
        action = environment.action_space.sample()
        observation, reward, terminated, truncated, info = environment.step(action)
        rollout_reward += float(reward)
        rollout_steps += 1
        if terminated or truncated:
            break
finally:
    environment.close()

assert rollout_steps > 0
print(
    f"Headless MetaDrive rollout passed: {rollout_steps} steps, "
    f"reward {rollout_reward:.3f}."
)

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Headless MetaDrive rollout passed: 25 steps, reward 0.349.


In [16]:
import json

report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "gpu": gpu_name,
    "python": sys.version.split()[0],
    "packages": package_versions,
    "metadrive_steps": rollout_steps,
    "metadrive_reward": rollout_reward,
}
report_dir = DRIVE_PROJECT / "smoke_tests"
report_dir.mkdir(parents=True, exist_ok=True)
report_name = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S.json")
report_path = report_dir / report_name
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"\nAll checks passed. Persistent report: {report_path}")

{
  "checked_at_utc": "2026-07-20T22:40:47.309353+00:00",
  "git_commit": "4ba4649f12937d61302f474b28d7c0b1ab6cf560",
  "gpu": "Tesla T4",
  "python": "3.12.13",
  "packages": {
    "metadrive-simulator": "0.4.3",
    "stable-baselines3": "2.9.0",
    "sb3-contrib": "2.9.0",
    "gymnasium": "1.3.0"
  },
  "metadrive_steps": 25,
  "metadrive_reward": 0.3488996429448679
}

All checks passed. Persistent report: /content/drive/MyDrive/SafeDrive/smoke_tests/20260720_224047.json


/usr/local/lib/python3.12/dist-packages/IPython/core/history.py:576: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.db.execute("""UPDATE sessions SET end=?, num_cmds=? WHERE


## Finished

If the report cell succeeded, VS Code, the Colab GPU runtime, GitHub, Google Drive, SafeDrive, and headless MetaDrive are connected correctly. Disconnect the Colab server when finished so it does not continue consuming compute units.

For later sessions, push local code changes to GitHub and rerun this notebook's clone/pull cell. Keep active training files on `/content` for speed, then copy checkpoints and completed runs to `MyDrive/SafeDrive`; do not treat the mounted Drive folder as a live Git checkout.